# Connectivity Notebook — Momentum Game-Theoretic Strategy
**Hugh Hayes · QUANTT**

Proof-of-access deliverable. Each cell makes a **real call** to one data source (or the local
stored data) and prints output proving the connection works. Doubles as evidence for the
**Data Source List** and the **Platform Decision**. Platorm decision was IBKR. Waiting on approval. When added, will add the api test here

Run top to bottom. Cell 0 sets the working directory so every relative path resolves.


## 0. Setup

## 1. Point-in-Time S&P 500 Membership

Daily snapshots of who was in the index, 1996 → 2026-01-14,
(Lehman, Enron, Wachovia, WaMu). Without this the backtest ranks today's 500 on past dates =
survivorship bias.


In [ ]:
import requests
import pandas as pd
import io
#simplified final version, final version uses existing csv, then the csv gets fed to the program and updated
WIKI_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

def scrape_last_n_changes(n=10):
    headers = {
        "User-Agent": "Mozilla/5.0 (QUANTT research script)",
        "Accept-Language": "en-US,en;q=0.9",
    }
    resp = requests.get(WIKI_URL, headers=headers, timeout=20)
    resp.raise_for_status()

    tables = pd.read_html(io.StringIO(resp.text))

    changes = None
    for t in tables:
        flat = [
            "|".join(map(str, c)) if isinstance(c, tuple) else str(c)
            for c in t.columns
        ]
        if any("Added" in c for c in flat) and any("Removed" in c for c in flat):
            changes = t
            break

    if changes is None:
        raise RuntimeError("Couldn't find the 'Selected changes' table on Wikipedia.")

    flat_cols = []
    for c in changes.columns:
        if isinstance(c, tuple):
            top, sub = c[0], c[1] if len(c) > 1 else ""
            if "Date" in top:
                flat_cols.append("date")
            elif "Added" in top and "Ticker" in sub:
                flat_cols.append("add_ticker")
            elif "Removed" in top and "Ticker" in sub:
                flat_cols.append("remove_ticker")
            else:
                flat_cols.append(f"{top}_{sub}")
        else:
            flat_cols.append(str(c))
    changes.columns = flat_cols

    changes = changes[["date", "add_ticker", "remove_ticker"]].copy()
    changes["date"] = pd.to_datetime(changes["date"], errors="coerce")
    for col in ("add_ticker", "remove_ticker"):
        changes[col] = (
            changes[col]
            .astype(str)
            .str.strip()
            .replace({"nan": "", "—": "", "-": ""})
        )
    changes = (
        changes.dropna(subset=["date"])
        .sort_values("date")
        .reset_index(drop=True)
    )

    return changes.tail(n)

print("Fetching S&P 500 changes from Wikipedia...")
df = scrape_last_n_changes(10)
print(f"\n[PASS] Scraped successfully — showing last 10 changes:\n")
print(df.to_string(index=False))
print(f"\nMost recent change date: {df['date'].max().date()}")

Fetching S&P 500 changes from Wikipedia...

[PASS] Scraped successfully — showing last 10 changes:

      date add_ticker remove_ticker
2025-12-22       CVNA          SOLS
2025-12-22        CRH           LKQ
2025-12-22        FIX           MHK
2026-02-09       CIEN           DAY
2026-03-23       SATS          PAYC
2026-03-23       COHR            LW
2026-03-23       LITE           MOH
2026-03-23        VRT          MTCH
2026-04-09       CASY          HOLX
2026-05-07       VEEV          CTRA

Most recent change date: 2026-05-07


## 2. Daily OHLCV - Yahoo Finance (yfinance)

Primary price feed. The big one-time pull (`pull_prices_historical.py`) used this to fetch every
historical member from 2003 → today into `data/prices_daily.parquet`. Here we just prove the
live connection with a simple call.

**Source:** yfinance / Yahoo Finance (free, no key).

In [ ]:
import yfinance as yf
#simple most recent call spy
spy = yf.download("SPY", period="1mo", progress=False, auto_adjust=True)
print("SPY rows pulled:", len(spy))
print(spy.tail(3))

SPY rows pulled: 21
Price            Close        High         Low        Open    Volume
Ticker             SPY         SPY         SPY         SPY       SPY
Date                                                                
2026-05-27  750.460022  751.380005  748.219971  750.880005  42106300
2026-05-28  754.599976  755.150024  749.229980  750.250000  41562600
2026-05-29  756.479980  758.080017  754.690002  755.900024  54976100


## 3. Stored Price Data — Local Parquet
In the data folder in local file

## 4. Risk-Free Rate — FRED (3-Month T-Bill, DGS3MO)

Used for the Sharpe ratio: `(strategy_return − rf) / vol`.

**Note:** pandas reading the URL directly fails with an SSL cert error on Python 3.14/macOS.
We pull the raw CSV with `requests` and `verify=False`, then parse from a string buffer.

In [ ]:
def fred_rf():
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS3MO"
    resp = requests.get(url, verify=False, timeout=15)
    resp.raise_for_status()
    df = pd.read_csv(io.StringIO(resp.text), parse_dates=["observation_date"])
    df.columns = ["date", "rf_3m_pct"]
    df["rf_3m_pct"] = pd.to_numeric(df["rf_3m_pct"], errors="coerce")
    return df.dropna()

rate = fred_rf()
print("Rows:", len(rate), "| latest:", rate["date"].max().date())
print(rate.tail(5).to_string(index=False))

/Users/hughhayes/Momentum Game Theory Trading Strategy/venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'fred.stlouisfed.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Rows: 11184 | latest: 2026-05-28
      date  rf_3m_pct
2026-05-21       3.68
2026-05-22       3.68
2026-05-26       3.68
2026-05-27       3.68
2026-05-28       3.69


## 5. Fama-French Factors + Momentum - Ken French Data Library

Factor regression to split strategy returns into known factor exposure vs genuine alpha.

**Note:** `pandas_datareader`'s famafrench reader is broken on Python 3.14 (a `deprecate_kwarg`
error), so we pull the raw ZIPs straight from Ken French's Dartmouth server with `requests` +
`zipfile`. Two gotchas handled: the file inside is named `.CSV` (uppercase), and the file
contains annual summary rows whose index isn't a `YYYYMM` date — we strip whitespace and keep
only rows matching `^\d{6}$`.

In [ ]:
import zipfile
def fetch_ff_zip(zip_name):
    base = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
    resp = requests.get(base + zip_name, verify=False, timeout=30)
    resp.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        # extension may be .CSV (uppercase) — match case-insensitively
        csv_name = next(n for n in zf.namelist() if n.upper().endswith(".CSV"))
        raw = zf.read(csv_name).decode("utf-8", errors="replace")

    # keep only lines that start with a digit (the monthly data block)
    rows = [l for l in raw.splitlines() if l.strip() and l.strip()[0].isdigit()]
    df = pd.read_csv(io.StringIO("\n".join(rows)), header=None, index_col=0)
    df.index = df.index.astype(str).str.strip()
    df = df[df.index.str.match(r"^\d{6}$")]              # drop annual summary rows
    df.index = pd.to_datetime(df.index, format="%Y%m") + pd.offsets.MonthEnd(0)
    return df.apply(pd.to_numeric, errors="coerce").dropna() / 100


ff = fetch_ff_zip("F-F_Research_Data_Factors_CSV.zip")
ff.columns = ["Mkt-RF", "SMB", "HML", "RF"]

mom = fetch_ff_zip("F-F_Momentum_Factor_CSV.zip")
mom.columns = ["Mom"]

factors = ff.join(mom, how="inner")
print("Factors:", list(factors.columns))
print("Range:", factors.index.min().date(), "->", factors.index.max().date())
print(factors.tail(3).to_string())

/Users/hughhayes/Momentum Game Theory Trading Strategy/venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mba.tuck.dartmouth.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Factors: ['Mkt-RF', 'SMB', 'HML', 'RF', 'Mom']
Range: 1927-01-31 -> 2026-04-30
            Mkt-RF     SMB     HML      RF     Mom
0                                                 
2026-02-28 -0.0117  0.0024  0.0265  0.0028  0.0127
2026-03-31 -0.0518  0.0044  0.0335  0.0029  0.0154
2026-04-30  0.0994  0.0013 -0.0127  0.0029  0.0962


/Users/hughhayes/Momentum Game Theory Trading Strategy/venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mba.tuck.dartmouth.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
